# Workbench Notebook

This notebook runs in the dedicated `workbench-jupyter` Conda environment.

Use the `wbn` helper below to call Workbench data-router endpoints without putting provider credentials in notebook code.

In [ ]:
import requests

try:
    import pandas as pd
except Exception:
    pd = None

class _Wbn:
    _BASE = "http://127.0.0.1:4000"

    def _query(self, moniker, shape, params=None):
        response = requests.post(
            f"{self._BASE}/api/data/query",
            json={"moniker": moniker, "shape": shape, "params": params or {}},
            timeout=20,
        )
        response.raise_for_status()
        return response.json().get("results", [])

    def _frame(self, rows):
        return pd.DataFrame(rows) if pd is not None else rows

    def fred(self, symbol, range="1y"):
        return self._frame(self._query(
            f"macro.indicators/{symbol}",
            "timeseries",
            {"symbol": symbol, "range": range},
        ))

    def equity(self, symbol, range="1y"):
        return self._frame(self._query(
            f"equity.prices/{symbol}",
            "timeseries",
            {"range": range},
        ))

    def curve(self):
        return self._frame(self._query("fixed.income.govies", "curve"))

    def rates(self):
        return self._frame(self._query("reference.rates", "snapshot"))

    def snapshot(self):
        return self._frame(self._query("macro.indicators", "snapshot"))

wbn = _Wbn()
print("wbn ready")

In [ ]:
wbn.fred("DGS10", range="3m").tail()